# Dask K-Means Multi-GPU Benchmark

This notebook provides a comprehensive benchmarking framework for cuML's Dask K-Means multi-GPU implementation. It allows you to:

- **Generate synthetic datasets** with configurable size (in GB), partitions, and dimensions
- **Load real data from CSV files** using cuDF (optional, disabled by default)
- **Benchmark performance** across multiple runs with detailed timing breakdowns
- **Visualize results** with performance charts

## Table of Contents

1. [Configuration Parameters](#configuration)
2. [Cluster Setup](#cluster_setup)
3. [Data Generation](#data_generation)
   - 3.1 [Synthetic Blobs Dataset](#synthetic_data)
   - 3.2 [CSV Data Loading (Optional)](#csv_data)
4. [Benchmark Execution](#benchmark)
5. [Results Analysis](#results)
6. [Cleanup](#cleanup)

## 1. Configuration Parameters <a id="configuration"/>

Adjust these parameters to customize the benchmark for your hardware and use case.

In [ ]:
# =============================================================================
# DATASET CONFIGURATION
# =============================================================================

# Data source: "synthetic" or "csv"
DATA_SOURCE = "synthetic"

# Synthetic dataset parameters
DATASET_SIZE_GB = 1.0            # Target dataset size in gigabytes
N_FEATURES = 50                  # Number of features/columns
N_PARTITIONS = None              # Number of partitions (None = auto, one per GPU)
DTYPE = "float32"                # Data type: "float32" or "float64"

# Synthetic blobs specific parameters
N_CENTERS = 10                   # Number of cluster centers for synthetic data
CLUSTER_STD = 0.5                # Standard deviation of clusters
RANDOM_STATE = 42                # Random seed for reproducibility

# CSV loading parameters (only used if DATA_SOURCE = "csv")
CSV_PATH = "./data/your_data.csv"  # Path to CSV file or glob pattern
CSV_COLUMNS = None               # List of columns to use (None = all numeric columns)
CSV_BLOCKSIZE = "256MB"          # Block size for reading CSV with Dask

# =============================================================================
# KMEANS CONFIGURATION
# =============================================================================

N_CLUSTERS = 10                  # Number of clusters to find
MAX_ITER = 300                   # Maximum number of iterations
TOL = 1e-4                       # Convergence tolerance
INIT_METHOD = "k-means||"        # Initialization: "k-means||" or "random"
OVERSAMPLING_FACTOR = 2.0        # Oversampling factor for k-means||

# =============================================================================
# BENCHMARK CONFIGURATION
# =============================================================================

N_WARMUP_RUNS = 1                # Number of warmup runs (not counted)
N_BENCHMARK_RUNS = 3             # Number of benchmark runs
VERBOSE = True                   # Print detailed progress

## Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import cupy as cp
import cudf
import dask_cudf

from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster

from cuml.dask.cluster import KMeans as DaskKMeans
from cuml.dask.datasets import make_blobs

import warnings
warnings.filterwarnings('ignore')

print(f"cudf version: {cudf.__version__}")

## Helper Functions

In [ ]:
def calculate_n_samples(size_gb, n_features, dtype="float32"):
    """
    Calculate the number of samples needed to achieve the target dataset size.
    
    Parameters
    ----------
    size_gb : float
        Target dataset size in gigabytes
    n_features : int
        Number of features/columns
    dtype : str
        Data type ("float32" or "float64")
    
    Returns
    -------
    int
        Number of samples
    """
    bytes_per_element = 4 if dtype == "float32" else 8
    bytes_total = size_gb * (1024 ** 3)
    n_samples = int(bytes_total / (n_features * bytes_per_element))
    return n_samples


def get_actual_size_gb(n_samples, n_features, dtype="float32"):
    """
    Calculate the actual dataset size in GB.
    """
    bytes_per_element = 4 if dtype == "float32" else 8
    return (n_samples * n_features * bytes_per_element) / (1024 ** 3)


def print_gpu_memory():
    """
    Print current GPU memory usage for all available GPUs.
    """
    try:
        import pynvml
        pynvml.nvmlInit()
        device_count = pynvml.nvmlDeviceGetCount()
        print("\nGPU Memory Usage:")
        print("-" * 50)
        for i in range(device_count):
            handle = pynvml.nvmlDeviceGetHandleByIndex(i)
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            name = pynvml.nvmlDeviceGetName(handle)
            if isinstance(name, bytes):
                name = name.decode('utf-8')
            used_gb = info.used / (1024**3)
            total_gb = info.total / (1024**3)
            print(f"  GPU {i} ({name}): {used_gb:.2f} GB / {total_gb:.2f} GB ({100*info.used/info.total:.1f}%)")
        pynvml.nvmlShutdown()
    except ImportError:
        print("pynvml not available - skipping GPU memory report")


class BenchmarkTimer:
    """
    Context manager for timing code blocks.
    """
    def __init__(self, name="", verbose=True):
        self.name = name
        self.verbose = verbose
        self.elapsed = 0
        
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        if self.verbose and self.name:
            print(f"  {self.name}: {self.elapsed:.4f}s")

## 2. Cluster Setup <a id="cluster_setup"/>

Initialize the Dask CUDA cluster with one worker per GPU.

In [ ]:
cluster = LocalCUDACluster(
    threads_per_worker=1,
    CUDA_VISIBLE_DEVICES=None,  # Or specify e.g. "0,1,2,3"
)
client = Client(cluster)

n_workers = len(client.scheduler_info()['workers'])
print(f"Dask cluster initialized with {n_workers} GPU worker(s)")
print(f"Dashboard: {client.dashboard_link}")

if N_PARTITIONS is None:
    N_PARTITIONS = n_workers
    print(f"Auto-set N_PARTITIONS = {N_PARTITIONS}")

print_gpu_memory()

## 3. Data Generation <a id="data_generation"/>

### 3.1 Synthetic Blobs Dataset <a id="synthetic_data"/>

Generate a synthetic dataset using cuML's `make_blobs` function. This creates clustered data directly on GPU memory.

In [ ]:
def generate_synthetic_data(size_gb, n_features, n_partitions, n_centers, 
                            cluster_std, dtype, random_state, verbose=True):
    """
    Generate synthetic blobs dataset distributed across GPUs.
    
    Returns
    -------
    X : dask_cudf.DataFrame or dask.array
        Feature matrix
    y : dask_cudf.Series or dask.array
        Cluster labels (ground truth)
    metadata : dict
        Dataset metadata
    """
    n_samples = calculate_n_samples(size_gb, n_features, dtype)
    actual_size_gb = get_actual_size_gb(n_samples, n_features, dtype)
    
    if verbose:
        print(f"\nGenerating synthetic dataset:")
        print(f"  Target size: {size_gb:.2f} GB")
        print(f"  Actual size: {actual_size_gb:.2f} GB")
        print(f"  Samples: {n_samples:,}")
        print(f"  Features: {n_features}")
        print(f"  Partitions: {n_partitions}")
        print(f"  Centers: {n_centers}")
        print(f"  Dtype: {dtype}")
    
    with BenchmarkTimer("Data generation time", verbose) as timer:
        X, y = make_blobs(
            n_samples=n_samples,
            n_features=n_features,
            centers=n_centers,
            n_parts=n_partitions,
            cluster_std=cluster_std,
            dtype=np.float32 if dtype == "float32" else np.float64,
            random_state=random_state,
            verbose=verbose
        )
        X = X.persist()
        y = y.persist()
        wait([X, y])
    
    metadata = {
        "source": "synthetic",
        "n_samples": n_samples,
        "n_features": n_features,
        "n_partitions": n_partitions,
        "size_gb": actual_size_gb,
        "dtype": dtype,
        "generation_time": timer.elapsed
    }
    
    return X, y, metadata

### 3.2 CSV Data Loading (Optional) <a id="csv_data"/>

Load data from CSV files using cuDF and Dask. Enable this by setting `DATA_SOURCE = "csv"` in the configuration.

In [ ]:
def load_csv_data(csv_path, columns=None, blocksize="256MB", verbose=True):
    """
    Load data from CSV file(s) using dask_cudf.
    
    Parameters
    ----------
    csv_path : str
        Path to CSV file or glob pattern (e.g., "./data/*.csv")
    columns : list or None
        List of column names to use. If None, uses all numeric columns.
    blocksize : str
        Size of each partition (e.g., "256MB")
    verbose : bool
        Print progress information
    
    Returns
    -------
    X : dask_cudf.DataFrame
        Feature matrix
    metadata : dict
        Dataset metadata
    """
    if verbose:
        print(f"\nLoading CSV data from: {csv_path}")
    
    with BenchmarkTimer("CSV loading time", verbose) as timer:
        ddf = dask_cudf.read_csv(csv_path, blocksize=blocksize)
        
        if columns is not None:
            ddf = ddf[columns]
        else:
            numeric_cols = ddf.select_dtypes(include=[np.number]).columns.tolist()
            if verbose:
                print(f"  Auto-selected {len(numeric_cols)} numeric columns")
            ddf = ddf[numeric_cols]
        
        ddf = ddf.astype("float32")
        X = ddf.persist()
        wait(X)
    
    n_samples = len(X)
    n_features = len(X.columns)
    n_partitions = X.npartitions
    size_gb = get_actual_size_gb(n_samples, n_features, "float32")
    
    if verbose:
        print(f"  Samples: {n_samples:,}")
        print(f"  Features: {n_features}")
        print(f"  Partitions: {n_partitions}")
        print(f"  Size: {size_gb:.2f} GB")
    
    metadata = {
        "source": "csv",
        "path": csv_path,
        "n_samples": n_samples,
        "n_features": n_features,
        "n_partitions": n_partitions,
        "size_gb": size_gb,
        "dtype": "float32",
        "loading_time": timer.elapsed
    }
    
    return X, metadata

### Load Data

Execute data loading based on the configured `DATA_SOURCE`.

In [ ]:
if DATA_SOURCE == "synthetic":
    X, y_true, data_metadata = generate_synthetic_data(
        size_gb=DATASET_SIZE_GB,
        n_features=N_FEATURES,
        n_partitions=N_PARTITIONS,
        n_centers=N_CENTERS,
        cluster_std=CLUSTER_STD,
        dtype=DTYPE,
        random_state=RANDOM_STATE,
        verbose=VERBOSE
    )
elif DATA_SOURCE == "csv":
    X, data_metadata = load_csv_data(
        csv_path=CSV_PATH,
        columns=CSV_COLUMNS,
        blocksize=CSV_BLOCKSIZE,
        verbose=VERBOSE
    )
    y_true = None
else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}. Use 'synthetic' or 'csv'.")

print_gpu_memory()

## 4. Benchmark Execution <a id="benchmark"/>

Run the K-Means benchmark with warmup and multiple timed runs.

In [ ]:
def run_kmeans_benchmark(X, n_clusters, max_iter, tol, init_method, 
                         oversampling_factor, n_warmup, n_runs, verbose=True):
    """
    Run K-Means benchmark. Uses persist() + wait() to keep data distributed
    across GPUs, allowing datasets larger than single GPU memory.
    """
    results = []
    
    if verbose:
        print(f"\nK-Means Configuration:")
        print(f"  Clusters: {n_clusters}")
        print(f"  Max iterations: {max_iter}")
        print(f"  Tolerance: {tol}")
        print(f"  Init method: {init_method}")
        print(f"  Warmup runs: {n_warmup}")
        print(f"  Benchmark runs: {n_runs}")
    
    if n_warmup > 0:
        if verbose:
            print(f"\n--- Warmup Runs ({n_warmup}) ---")
        for i in range(n_warmup):
            if verbose:
                print(f"Warmup run {i+1}/{n_warmup}...")
            model = DaskKMeans(
                n_clusters=n_clusters,
                max_iter=max_iter,
                tol=tol,
                init=init_method,
                oversampling_factor=oversampling_factor,
                random_state=RANDOM_STATE
            )
            model.fit(X)
            del model
    
    if verbose:
        print(f"\n--- Benchmark Runs ({n_runs}) ---")
    
    for i in range(n_runs):
        if verbose:
            print(f"\nRun {i+1}/{n_runs}:")
        
        run_result = {"run": i + 1}
        
        model = DaskKMeans(
            n_clusters=n_clusters,
            max_iter=max_iter,
            tol=tol,
            init=init_method,
            oversampling_factor=oversampling_factor,
            random_state=RANDOM_STATE + i
        )
        
        with BenchmarkTimer("Fit time", verbose) as fit_timer:
            model.fit(X)
        run_result["fit_time"] = fit_timer.elapsed
        
        with BenchmarkTimer("Predict time", verbose) as predict_timer:
            labels = model.predict(X).persist()
            wait(labels)
        run_result["predict_time"] = predict_timer.elapsed
        
        with BenchmarkTimer("Transform time", verbose) as transform_timer:
            distances = model.transform(X).persist()
            wait(distances)
        run_result["transform_time"] = transform_timer.elapsed
        
        run_result["total_time"] = fit_timer.elapsed + predict_timer.elapsed + transform_timer.elapsed
        run_result["inertia"] = float(model.inertia_)
        run_result["n_iter"] = int(model.n_iter_)
        
        results.append(run_result)
        
        if i < n_runs - 1:
            del labels, distances, model
    
    return results, model, labels

In [ ]:
benchmark_results, final_model, final_labels = run_kmeans_benchmark(
    X=X,
    n_clusters=N_CLUSTERS,
    max_iter=MAX_ITER,
    tol=TOL,
    init_method=INIT_METHOD,
    oversampling_factor=OVERSAMPLING_FACTOR,
    n_warmup=N_WARMUP_RUNS,
    n_runs=N_BENCHMARK_RUNS,
    verbose=VERBOSE
)

print_gpu_memory()

## 5. Results Analysis <a id="results"/>

In [ ]:
results_df = pd.DataFrame(benchmark_results)
results_df["n_samples"] = data_metadata["n_samples"]
results_df["n_features"] = data_metadata["n_features"]
results_df["n_partitions"] = data_metadata["n_partitions"]
results_df["size_gb"] = data_metadata["size_gb"]
results_df["n_clusters"] = N_CLUSTERS
results_df["n_gpus"] = n_workers

print("\n" + "="*60)
print("BENCHMARK RESULTS")
print("="*60)
results_df

In [ ]:
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

summary = results_df[["fit_time", "predict_time", "transform_time", "total_time"]].agg(["mean", "std", "min", "max"])
print(summary.round(4))

mean_fit_time = results_df["fit_time"].mean()
samples_per_second = data_metadata["n_samples"] / mean_fit_time
gb_per_second = data_metadata["size_gb"] / mean_fit_time

print(f"\nThroughput (based on mean fit time):")
print(f"  {samples_per_second:,.0f} samples/second")
print(f"  {gb_per_second:.2f} GB/second")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
timing_cols = ["fit_time", "predict_time", "transform_time"]

ax1 = axes[0]
results_df[timing_cols].plot(kind="bar", ax=ax1)
ax1.set_xlabel("Run")
ax1.set_ylabel("Time (seconds)")
ax1.set_title("Timing Breakdown by Run")
ax1.set_xticklabels([f"Run {i+1}" for i in range(len(results_df))], rotation=0)
ax1.legend(loc="upper right")

ax2 = axes[1]
results_df[timing_cols].plot(kind="bar", stacked=True, ax=ax2, color=["#2ecc71", "#3498db", "#9b59b6"])
ax2.set_xlabel("Run")
ax2.set_ylabel("Time (seconds)")
ax2.set_title("Total Time Composition")
ax2.set_xticklabels([f"Run {i+1}" for i in range(len(results_df))], rotation=0)
ax2.legend(loc="upper right")

ax3 = axes[2]
ax3.bar(range(1, len(results_df) + 1), results_df["inertia"], color="#e74c3c", alpha=0.7)
ax3.set_xlabel("Run")
ax3.set_ylabel("Inertia")
ax3.set_title("Final Inertia by Run")
ax3.set_xticks(range(1, len(results_df) + 1))

plt.tight_layout()
plt.show()

In [ ]:
if y_true is not None:
    from cuml.metrics import adjusted_rand_score
    
    print("\n" + "="*60)
    print("CLUSTERING QUALITY (vs Ground Truth)")
    print("="*60)
    
    # compute() here is only for quality metrics, not part of benchmark timing
    labels_pred = final_labels.compute()
    y_true_computed = y_true.compute()
    
    ari_score = adjusted_rand_score(y_true_computed, labels_pred)
    print(f"Adjusted Rand Index: {ari_score:.4f}")
    print(f"(1.0 = perfect clustering, 0.0 = random)")

In [ ]:
output_filename = f"kmeans_mnmg_benchmark_{data_metadata['n_samples']}samples_{n_workers}gpus.csv"
results_df.to_csv(output_filename, index=False)
print(f"\nResults exported to: {output_filename}")

## 6. Cleanup <a id="cleanup"/>

In [ ]:
# Clean up GPU memory
del X
if y_true is not None:
    del y_true
del final_model, final_labels

In [ ]:
# Shutdown cluster 
client.close()
cluster.close()
print("Cluster shutdown complete.")

---

## Quick Reference: CSV Data Loading

To use your own CSV data instead of synthetic data, modify the configuration cell:

```python
# Change data source
DATA_SOURCE = "csv"

# Set your CSV path (supports glob patterns)
CSV_PATH = "./data/my_dataset.csv"        # Single file
# CSV_PATH = "./data/my_dataset_*.csv"    # Multiple files with glob
# CSV_PATH = "/path/to/s3/bucket/*.csv"   # S3 path (requires s3fs)

# Optionally specify columns (None = auto-select numeric columns)
CSV_COLUMNS = ["feature_1", "feature_2", "feature_3"]  # Or None

# Adjust block size for memory management
CSV_BLOCKSIZE = "256MB"  # Larger = fewer partitions, smaller = more partitions
```

### Loading from Multiple Sources

For advanced use cases, you can modify `load_csv_data()` to support:
- **Parquet files**: Use `dask_cudf.read_parquet()` instead of `read_csv()`
- **JSON files**: Use `dask_cudf.read_json()`
- **S3/GCS/Azure**: Install `s3fs`, `gcsfs`, or `adlfs` and use cloud URLs